# Agentic RAG: Router–Retriever System (Anthropic + Docling + Local Embeddings)

This notebook implements a multi-agent Retrieval-Augmented Generation (RAG) system using **CrewAI**, running entirely on **Claude** for generation and **local HuggingFace embeddings** for retrieval.

## Workflow

| Stage | Tool |
|-------|--------------|
| Agent LLM | **Claude** (`ANTHROPIC_API_KEY`) |
| PDF reading | **Docling** structured parser |
| Embeddings | **Local** `all-MiniLM-L6-v2` (no key, no quota) |
| Vector store | **ChromaDB** (explicit, inspectable) |
| Web search | Tavily |


## System Architecture

```
User Query
    │
    ▼
┌─────────────────────────────────┐
│ Router Agent  (Claude)          │
│ Classifies query into:          │
│   - pdf_search                  │
│   - web_search                  │
│   - generate                    │
└──────────────┬──────────────────┘
               │ routing decision
               ▼
┌─────────────────────────────────┐
│ Retriever Agent  (Claude)       │
│ Executes retrieval via:         │
│   - Docling+Chroma PDF tool     │
│   - Tavily live web search      │
│   - LLM knowledge (generate)    │
└──────────────┬──────────────────┘
               │ grounded answer
               ▼
          Final Answer
```


## 1. Install Dependencies

Docling pulls in some heavyweight dependencies (PyTorch, layout models) — the install cell takes a minute or two on a fresh Colab runtime.

In [3]:
# Core: CrewAI + Anthropic routing, Docling parser, local embeddings, Chroma, Tavily
!pip install -q \
  "crewai" \
  "crewai-tools" \
  "docling" \
  "langchain-huggingface" \
  "sentence-transformers" \
  "chromadb" \
  "langchain-community" \
  "tavily-python" \
  "anthropic" \
  "langchain-text-splitters"

print("\n✅ Dependencies installed. If this is Colab, restart the runtime now:")
print("   Runtime ▸ Restart session, then run from Section 2.")
print("   (A restart clears the stale module cache after big pip installs.)")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 8.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 6.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 56.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.1/223.1 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 75.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 841.5/841.5 kB 56.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 974.7/974.7 kB 62.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.9/19.9 MB 79.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 91.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/

> **⚠️ If running in Colab:** After the install above finishes, do **Runtime ▸ Restart session** before continuing. A `ModuleNotFoundError` for `langchain_huggingface` almost always means a skipped install cell or a stale kernel — restarting fixes it. You do *not* need to re-run the install after restarting.

## 2. Environment Setup & API Keys

This part reads secrets from **Colab Secrets** (the 🔑 panel on the left), not a `.env` file.

Add two secrets and toggle **Notebook access** on for each:

| Secret name | Used for | Required? |
|-------------|----------|-----------|
| `ANTHROPIC_API_KEY` | Claude (both agents) | **Yes** |
| `TAVILY_API_KEY` | Live web search path | Only if you demo `web_search` |


In [1]:
import os

# In Colab, pull secrets from userdata. Locally, fall back to existing env vars.
try:
  from google.colab import userdata

  def _load(name, required=True):
    try:
      val = userdata.get(name)
    except Exception:
      val = None
    if val:
      os.environ[name] = val
    elif required:
      print(f"⚠️  Secret '{name}' not found in Colab Secrets (🔑 panel).")
    return val

  _load("ANTHROPIC_API_KEY", required=True)
  _load("TAVILY_API_KEY", required=False)
  _env = "Colab Secrets"
except ImportError:
  # Not on Colab — assume keys are already exported in the shell environment.
  _env = "local environment variables"

assert os.environ.get("ANTHROPIC_API_KEY"), (
  "ANTHROPIC_API_KEY is not set. Add it to Colab Secrets (🔑) and enable "
  "Notebook access, then re-run this cell."
)

# Path to the PDF the retriever will index. Upload it to the Colab file browser,
# or adjust this path. The transformer paper is a good demo doc — dense with
# tables and multi-column layout, which is exactly where Docling shines.
PDF_PATH = "data/transformer_research_paper-dataset.pdf"

print(f"✅ Environment configured using {_env}.")
print(f"   ANTHROPIC_API_KEY: {'set' if os.environ.get('ANTHROPIC_API_KEY') else 'MISSING'}")
print(f"   TAVILY_API_KEY:    {'set' if os.environ.get('TAVILY_API_KEY') else 'not set (web_search disabled)'}")

✅ Environment configured using Colab Secrets.
   ANTHROPIC_API_KEY: set
   TAVILY_API_KEY:    set


## 3. Import Libraries & Set Up Logging

The logging block writes a full reasoning trace to both a file and stdout.

In [2]:
import logging
import json
from datetime import datetime, timezone
from textwrap import dedent

from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import BaseTool
from pydantic import Field

# ── Logging setup ────────────────────────────────────────────────
LOG_FILE = "agentic_rag_trace.log"
logging.basicConfig(
  level=logging.INFO,
  format="%(asctime)s [%(levelname)s] %(message)s",
  handlers=[
    logging.FileHandler(LOG_FILE, mode="w"),
    logging.StreamHandler(),
  ],
  force=True,  # Colab pre-configures logging; without this the call is ignored and the log file stays empty
)
logger = logging.getLogger("AgenticRAG")
logger.info("Logger initialised — Agentic RAG session started.")
print("✅ Imports and logging ready.")

2026-10-07 07:42:03,054 [INFO] Logger initialised — Agentic RAG session started.


✅ Imports and logging ready.


## 4. Configure the Claude LLM for both agents

CrewAI talks to model providers through LiteLLM, so pointing it at Claude is a one-line config change. We wrap it in a `crewai.LLM` object and hand the *same* instance to both agents.

The model string uses the `anthropic/` provider prefix that LiteLLM expects.

In [17]:
# One Claude instance, shared by both agents.
# No temperature: the current Anthropic Python SDK no longer accepts sampling
# parameters in messages.create(), so passing one raises a TypeError.
claude_llm = LLM(
  model="anthropic/claude-sonnet-4-5-20250929"
)

logger.info("Claude LLM configured: %s", claude_llm.model)
print(f"✅ Claude LLM ready: {claude_llm.model}")

2026-10-07 07:57:49,852 [INFO] Claude LLM configured: claude-sonnet-4-5-20250929


✅ Claude LLM ready: claude-sonnet-4-5-20250929


## 5. Build the PDF retrieval pipeline (Docling → chunk → local embed → Chroma)


**5a. Parse with Docling.** Docling does document *structure* recognition — tables, headings, reading order — not just a flat text dump. For a multi-column research paper that means the extracted Markdown chunks cleanly along real section boundaries instead of splicing columns together.

In [4]:
from docling.document_converter import DocumentConverter

def parse_pdf_to_markdown(pdf_path: str) -> str:
  """Parse a PDF into structure-aware Markdown using Docling (v2 API)."""
  logger.info("[Docling] Parsing PDF: %s", pdf_path)
  converter = DocumentConverter()
  result = converter.convert(pdf_path)          # v2 API: convert() ...
  markdown = result.document.export_to_markdown()  # ... then export_to_markdown()
  logger.info("[Docling] Parsed %d characters of Markdown.", len(markdown))
  return markdown

# First Docling run downloads its layout models (~tens of MB) and takes ~30-60s.
pdf_markdown = parse_pdf_to_markdown(PDF_PATH)
print(f"✅ Docling parsed the PDF into {len(pdf_markdown):,} characters of Markdown.")
print("\n─── First 600 chars preview ───")
print(pdf_markdown[:600])

2026-10-07 07:42:10,033 [INFO] NumExpr defaulting to 2 threads.
2026-10-07 07:42:21,848 [INFO] [Docling] Parsing PDF: transformer_research_paper-dataset.pdf
2026-10-07 07:42:22,496 [INFO] detected formats: [<InputFormat.PDF: 'pdf'>]
2026-10-07 07:42:22,588 [INFO] Going to convert document batch...
2026-10-07 07:42:22,590 [INFO] Initializing pipeline for StandardPdfPipeline with options hash b023c96447cfca452ff1bd014d549b6e
2026-10-07 07:42:22,678 [INFO] Loading plugin 'docling_defaults'
2026-10-07 07:42:22,681 [INFO] Registered picture descriptions: ['picture_description_vlm_engine', 'vlm', 'api']
2026-10-07 07:42:22,767 [INFO] Loading plugin 'docling_defaults'
2026-10-07 07:42:22,795 [INFO] Registered ocr engines: ['auto', 'easyocr', 'kserve_v2_ocr', 'nemotron-ocr', 'ocrmac', 'rapidocr', 'tesserocr', 'tesseract']
2026-10-07 07:42:22,798 [INFO] Nemotron cannot be used because it is not installed.
2026-10-07 07:42:23,960 [INFO] Accelerator device: 'cuda:0'
[INFO] 2026-10-07 07:42:23,999

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

2026-10-07 07:42:37,319 [INFO] Transformers engine ready (device=cuda:0, dtype=torch.float32)
2026-10-07 07:42:37,393 [INFO] Loading plugin 'docling_defaults'
2026-10-07 07:42:37,398 [INFO] Registered table structure engines: ['docling_tableformer', 'docling_tableformer_v2', 'granite_vision_table']
2026-10-07 07:42:37,399 [INFO] Fetching model docling-project/docling-models (revision: v2.3.0)...
2026-10-07 07:42:37,525 [INFO] HTTP Request: GET https://huggingface.co/api/models/docling-project/docling-models/revision/v2.3.0 "HTTP/1.1 200 OK"
2026-10-07 07:42:37,634 [INFO] HTTP Request: GET https://huggingface.co/api/models/docling-project/docling-models/tree/fc0f2d45e2218ea24bce5045f58a389aed16dc23?recursive=true&expand=false "HTTP/1.1 200 OK"
2026-10-07 07:42:37,739 [INFO] HTTP Request: HEAD https://huggingface.co/docling-project/docling-models/resolve/fc0f2d45e2218ea24bce5045f58a389aed16dc23/.gitattributes "HTTP/1.1 307 Temporary Redirect"
2026-10-07 07:42:37,751 [INFO] HTTP Request: 

✅ Docling parsed the PDF into 48,951 characters of Markdown.

─── First 600 chars preview ───
Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works.

## Attention Is All You Need

Ashish Vaswani∗ Google Brain avaswani@google.com Noam Shazeer∗ Google Brain noam@google.com Niki Parmar∗ Google Research nikip@google.com Jakob Uszkoreit∗ Google Research usz@google.com

Llion Jones∗ Google Research llion@google.com Aidan N. Gomez∗ † University of Toronto aidan@cs.toronto.edu Łukasz Kaiser∗ Google Brain lukaszkaiser@google.com Illia Polosukhin∗ ‡ illia.polosukhin@gmail.com

#


**5a-ii. Tell the Router what's in the PDF.** The Router can't see the document, so without this it only knows "there is a PDF". For a well-known topic like attention, it then decides it can answer from memory and picks `generate`. Here we build a short table of contents from Docling's headings and give it to the Router, so it knows which questions the document covers.

In [5]:
import re

PDF_TITLE = "Attention Is All You Need (Vaswani et al., 2017), the paper that introduced the Transformer"

# Section headings from Docling's Markdown, e.g. "3.2 Attention", "6 Results"
_headings = re.findall(r"^#{1,3}[ \t]+(.+?)[ \t]*$", pdf_markdown, flags=re.MULTILINE)
_headings = [h for h in dict.fromkeys(_headings) if len(h) > 2]  # dedupe, keep order
PDF_TOPICS = "; ".join(_headings[:30])

print("PDF:", PDF_TITLE)
print("Sections the Router will see:\n ", PDF_TOPICS)

PDF: Attention Is All You Need (Vaswani et al., 2017), the paper that introduced the Transformer
Sections the Router will see:
  Attention Is All You Need; Abstract; 1 Introduction; 2 Background; 3 Model Architecture; 3.1 Encoder and Decoder Stacks; 3.2 Attention; Scaled Dot-Product Attention; 3.2.1 Scaled Dot-Product Attention; 3.2.2 Multi-Head Attention; 3.2.3 Applications of Attention in our Model; 3.3 Position-wise Feed-Forward Networks; 3.4 Embeddings and Softmax; 3.5 Positional Encoding; 4 Why Self-Attention; 5 Training; 5.1 Training Data and Batching; 5.2 Hardware and Schedule; 5.3 Optimizer; 5.4 Regularization; 6 Results; 6.1 Machine Translation; 6.2 Model Variations; 6.3 English Constituency Parsing; 7 Conclusion; References; Input-Input Layer5


**5b. Chunk.** Split the Markdown into overlapping passages. Overlap keeps a sentence that straddles a boundary from being cut in half. We chunk on the Markdown so structure is respected.

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
  chunk_size=1000,
  chunk_overlap=150,
  separators=["\n## ", "\n### ", "\n\n", "\n", ". ", " ", ""],
)
chunks = splitter.split_text(pdf_markdown)
logger.info("[Chunk] Split Markdown into %d chunks.", len(chunks))
print(f"✅ Split into {len(chunks)} chunks (size≈1000, overlap=150).")

2026-10-07 07:44:14,038 [INFO] [Chunk] Split Markdown into 71 chunks.


✅ Split into 71 chunks (size≈1000, overlap=150).


**5c. Embed locally + store in Chroma.** The embedding model runs on the Colab processor — no API key, no quota, no remote endpoint.

In [7]:
import chromadb
from langchain_huggingface import HuggingFaceEmbeddings

# Local embedding model — downloads once (~90 MB), then runs offline on CPU.
embedder = HuggingFaceEmbeddings(
  model_name="sentence-transformers/all-MiniLM-L6-v2",
)
logger.info("[Embed] Local embedding model loaded: all-MiniLM-L6-v2")

# In-memory Chroma collection (resets each session — fine for a demo).
chroma_client = chromadb.Client()
COLLECTION_NAME = "pdf_chunks"
# Drop any existing collection so re-running the cell is idempotent.
try:
  chroma_client.delete_collection(COLLECTION_NAME)
except Exception:
  pass
collection = chroma_client.create_collection(COLLECTION_NAME)

# Embed all chunks locally and add them to Chroma.
embeddings = embedder.embed_documents(chunks)
collection.add(
  ids=[f"chunk_{i}" for i in range(len(chunks))],
  documents=chunks,
  embeddings=embeddings,
)
logger.info("[Chroma] Stored %d embedded chunks.", len(chunks))
print(f"✅ Embedded and stored {len(chunks)} chunks in Chroma (local, no API key).")

2026-10-07 07:44:18,151 [INFO] TensorFlow version 2.21.0 available.
2026-10-07 07:44:18,154 [INFO] JAX version 0.11.1 available.
2026-10-07 07:44:21,961 [INFO] No device provided, using cuda:0
2026-10-07 07:44:22,089 [INFO] HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-10-07 07:44:22,099 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/modules.json?%2Fsentence-transformers%2Fall-MiniLM-L6-v2%2Fresolve%2Fmain%2Fmodules.json=&etag=%22952a9b81c0bfd99800fabf352f69c7ccd46c5e43%22 "HTTP/1.1 200 OK"
2026-10-07 07:44:22,110 [INFO] HTTP Request: GET https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/modules.json?%2Fsentence-transformers%2Fall-MiniLM-L6-v2%2Fresolve%2Fmain%2Fmodules.json=&etag=%22952a9b81c0bfd99800fabf

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

2026-10-07 07:44:25,493 [INFO] HTTP Request: GET https://huggingface.co/api/models/sentence-transformers/all-MiniLM-L6-v2 "HTTP/1.1 200 OK"
2026-10-07 07:44:25,596 [INFO] HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/processor_config.json "HTTP/1.1 404 Not Found"
2026-10-07 07:44:25,696 [INFO] HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-10-07 07:44:25,796 [INFO] HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/video_preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-10-07 07:44:25,897 [INFO] HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-10

✅ Embedded and stored 71 chunks in Chroma (local, no API key).


**5d. Wrap search as a CrewAI tool.** Finally we expose similarity search as a `BaseTool` so the Retriever agent can call it — the same interface the original `PDFSearchTool` presented to the agent, but now backed by our own transparent pipeline.

In [8]:
class DoclingPDFSearchTool(BaseTool):
  """Semantic search over the Docling-parsed, locally-embedded PDF chunks."""
  name: str = "pdf_search"
  description: str = (
    "Searches the local PDF, 'Attention Is All You Need' (the original "
    "Transformer paper), for passages about its architecture, attention, "
    "training setup and results. Use this for questions on those topics."
  )

  def _run(self, query: str) -> str:
    logger.info("[DoclingPDFSearchTool] Query: %s", query)
    q_emb = embedder.embed_query(query)
    results = collection.query(query_embeddings=[q_emb], n_results=4)
    docs = results.get("documents", [[]])[0]
    if not docs:
      return "No relevant passages found in the PDF."
    logger.info("[DoclingPDFSearchTool] Returned %d passages.", len(docs))
    return "\n\n---\n\n".join(
      f"[Passage {i+1}]\n{d}" for i, d in enumerate(docs)
    )

pdf_search_tool = DoclingPDFSearchTool()
logger.info("DoclingPDFSearchTool initialised.")
print("✅ PDF search tool ready (Docling + local embeddings + Chroma).")

2026-10-07 07:44:45,497 [INFO] DoclingPDFSearchTool initialised.


✅ PDF search tool ready (Docling + local embeddings + Chroma).


## 6. Web Search Tool (Tavily)

CrewAI-compatible wrapper around Tavily is the one path that needs a paid key (`TAVILY_API_KEY`). If you skipped that secret, the `web_search` route will raise when exercised; the `pdf_search` and `generate` routes still work.

In [9]:
from langchain_community.tools.tavily_search import TavilySearchResults

class TavilyWebSearchTool(BaseTool):
  """CrewAI-compatible wrapper around TavilySearchResults."""
  name: str = "web_search"
  description: str = (
    "Searches the internet for up-to-date information. Use this when the "
    "question requires recent or real-world knowledge not in the local PDF."
  )
  _client: TavilySearchResults = None

  def model_post_init(self, __context):
    self._client = TavilySearchResults(max_results=5)

  def _run(self, query: str) -> str:
    logger.info("[TavilyWebSearchTool] Query: %s", query)
    results = self._client.invoke(query)
    formatted = [
      f"Source: {r.get('url', 'N/A')}\nSnippet: {r.get('content', '')}"
      for r in results
    ]
    logger.info("[TavilyWebSearchTool] Returned %d results.", len(results))
    return "\n\n".join(formatted)

web_search_tool = TavilyWebSearchTool()
logger.info("TavilyWebSearchTool initialised.")
print("✅ Both retrieval tools are ready.")

/tmp/ipykernel_2705/1740401033.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools.tavily_search import TavilySearchResults
/tmp/ipykernel_2705/1740401033.py:13: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  self._client = TavilySearchResults(max_results=5)
2026-10-07 07:44:46,898 [INFO] TavilyWebSearchTool initialised.


✅ Both retrieval tools are ready.


## 7. Define the Agents

Two agents, both backed by the shared Claude LLM. Note the explicit `llm=claude_llm` on each — that's what routes them to Claude instead of CrewAI's default.

**Router Agent** — classifies the query into `pdf_search`, `web_search`, or `generate`.
**Retriever Agent** — acts on the routing decision, calls the right tool, and composes a grounded answer.

In [10]:
# ── 7a. Router Agent ──────────────────────────────────────────────
router_agent = Agent(
  role="Query Router",
  goal=dedent(f"""\
    Analyse the user's question and decide the most appropriate retrieval strategy.
    The local PDF is: {PDF_TITLE}.
    Its sections: {PDF_TOPICS}
    Output ONLY one of the following labels (no extra text), applying these rules IN ORDER:
      1. pdf_search : the question's topic is covered by the PDF above. Choose this EVEN IF
                      you could answer from general knowledge; answers must be grounded in
                      the document whenever it covers the topic.
      2. web_search : the question needs current, recent, or real-world information
                      that the PDF cannot contain.
      3. generate   : anything else that general knowledge can answer.
  """),
  backstory=dedent("""\
    You are an expert information architect specialising in query intent classification.
    You know when static document retrieval, live web search, or pure language-model
    generation is the right approach. You are concise and always produce exactly one label.
  """),
  llm=claude_llm,
  verbose=True,
  allow_delegation=False,
)
logger.info("Router Agent created.")

# ── 7b. Retriever Agent ───────────────────────────────────────────
retriever_agent = Agent(
  role="Information Retriever",
  goal=dedent("""\
    Given a routing label and the original user question, retrieve relevant information
    using the correct tool and compose a clear, accurate, source-grounded answer.
      - If the label is 'pdf_search' -> use the pdf_search tool.
      - If the label is 'web_search' -> use the web_search tool.
      - If the label is 'generate'   -> answer directly from your own knowledge.
    Always cite the source of your information in the final answer.
  """),
  backstory=dedent("""\
    You are a meticulous research analyst who extracts precise information from documents
    and the web. You ground answers in retrieved evidence and clearly indicate the source.
  """),
  tools=[pdf_search_tool, web_search_tool],
  llm=claude_llm,
  verbose=True,
  allow_delegation=False,
)
logger.info("Retriever Agent created.")
print("✅ Both agents defined (backed by Claude).")

2026-10-07 07:44:48,625 [INFO] Router Agent created.
2026-10-07 07:44:48,628 [INFO] Retriever Agent created.


✅ Both agents defined (backed by Claude).


## 8. Define the Tasks

The routing task feeds its output to the retrieval task as context — the reason the crew runs sequentially rather than in parallel.

In [11]:
def build_tasks(user_question: str):
  """Construct routing and retrieval tasks for a given user question."""
  routing_task = Task(
    description=dedent(f"""\
      A user has asked the following question:
      "{user_question}"

      The local PDF is: {PDF_TITLE}.
      Its sections: {PDF_TOPICS}

      Classify this question and respond with EXACTLY ONE of these labels, checking IN ORDER:
        pdf_search  — the PDF covers this topic (prefer this even if you already know the answer)
        web_search  — the answer requires current internet information
        generate    — none of the above; general knowledge is enough

      Do NOT include any explanation, punctuation, or extra words — just the label.
    """),
    expected_output="One of: pdf_search | web_search | generate",
    agent=router_agent,
  )

  retrieval_task = Task(
    description=dedent(f"""\
      The Router Agent has provided a routing decision (see context above).
      The original user question is:
      "{user_question}"

      Follow the routing decision precisely:
        - pdf_search -> call the pdf_search tool with the question as input
        - web_search -> call the web_search tool with the question as input
        - generate   -> answer directly from your LLM knowledge

      Compose a comprehensive, well-structured answer.
      Clearly state which source (PDF, Web, or LLM knowledge) was used.
    """),
    expected_output="A clear, source-grounded answer to the user's question.",
    agent=retriever_agent,
    context=[routing_task],  # passes Router output as context
  )
  return routing_task, retrieval_task

print("✅ Task builder function ready.")

✅ Task builder function ready.


## 9. Assemble the Crew & Run Pipeline

In [12]:
import re

async def run_agentic_rag(user_question: str) -> dict:
  """Execute the full Agentic RAG pipeline for a single user question."""
  logger.info("=" * 60)
  logger.info("New query received: %s", user_question)

  routing_task, retrieval_task = build_tasks(user_question)
  crew = Crew(
    agents=[router_agent, retriever_agent],
    tasks=[routing_task, retrieval_task],
    process=Process.sequential,  # Router first, then Retriever
    verbose=True,
  )
  # Notebooks already run an event loop, so CrewAI requires the async kickoff here.
  result = await crew.kickoff_async()

  raw_route = routing_task.output.raw.lower() if routing_task.output else ""
  match = re.search(r"pdf_search|web_search|generate", raw_route)
  route_label = match.group(0) if match else "unknown"
  final_answer = result.raw if hasattr(result, "raw") else str(result)

  logger.info("Route chosen : %s", route_label)
  preview = final_answer[:300] + ("..." if len(final_answer) > 300 else "")
  logger.info("Final answer : %s", preview)
  logger.info("=" * 60)

  return {
    "question": user_question,
    "route": route_label,
    "answer": final_answer,
    "timestamp": datetime.now(timezone.utc).isoformat(),
  }

print("✅ run_agentic_rag() function ready.")

✅ run_agentic_rag() function ready.


## 10. Demo — Run Three Example Queries

| Query | Expected Route |
|-------|----------------|
| How does multi-head attention work in the Transformer, and why use multiple heads? | `pdf_search` |
| What are the latest breakthroughs in AI announced this week? | `web_search` |
| What is the capital of France? | `generate` |


In [13]:
demo_questions = [
  "How does multi-head attention work in the Transformer, and why use multiple heads?",  # -> pdf_search
  "What are the latest breakthroughs in AI announced this week?",  # -> web_search
  "What is the capital of France?",                          # -> generate
]

results = []
for i, question in enumerate(demo_questions, 1):
  print(f"\n{'='*70}")
  print(f"QUERY {i}: {question}")
  print('='*70)
  output = await run_agentic_rag(question)
  results.append(output)
  print(f"\n📍 Route  : {output['route']}")
  ans = output['answer']
  print(f"💬 Answer : {ans[:500]}..." if len(ans) > 500 else f"💬 Answer : {ans}")

2026-10-07 07:44:55,179 [INFO] ============================================================
2026-10-07 07:44:55,180 [INFO] New query received: How does multi-head attention work in the Transformer, and why use multiple heads?



QUERY 1: How does multi-head attention work in the Transformer, and why use multiple heads?


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: ddd5c9a1-8509-48b0-8fe0-032251599512                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: A user has asked the following question:                                                                 │
│  "How does multi-head attention work in the Transformer, and why use multiple heads?"                           │
│                                                                                                                 │
│  The local PDF is: Attention Is All You Need (Vaswani et al., 2017), the paper that introduced the              │
│  Transformer.                                                                                                   │
│  Its sections: Attention Is All You Need; Abstract; 1 Introduction; 2 Background; 3 Model Architecture; 3.1     │
│  Encoder and Decoder Stacks; 3.2 Attention; Scaled Dot-Product Attention; 3.2.1 Scaled Dot-Product Attention;   │
│  3.2.2 Multi-Head Attention; 3.2.3 Applications of Attention in our Model; 3.3 Position-wise Feed-Forward       │
│  Networks; 3.4 Embeddings and Softmax; 3.5 Positional Encoding; 4 Why Self-Attention; 5 Training; 5.1 Training  │
│  Data and Batching; 5.2 Hardware and Schedule; 5.3 Optimizer; 5.4 Regularization; 6 Results; 6.1 Machine        │
│  Translation; 6.2 Model Variations; 6.3 English Constituency Parsing; 7 Conclusion; References; Input-Input     │
│  Layer5                                                                                                         │
│                                                                                                                 │
│  Classify this question and respond with EXACTLY ONE of these labels, checking IN ORDER:                        │
│    pdf_search  — the PDF covers this topic (prefer this even if you already know the answer)                    │
│    web_search  — the answer requires current internet information                                               │
│    generate    — none of the above; general knowledge is enough                                                 │
│                                                                                                                 │
│  Do NOT include any explanation, punctuation, or extra words — just the label.                                  │
│                                                                                                                 │
│  ID: d57a6e40-6e15-4bf8-8a85-e3db70ce4bc2                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Query Router                                                                                            │
│                                                                                                                 │
│  Task: A user has asked the following question:                                                                 │
│  "How does multi-head attention work in the Transformer, and why use multiple heads?"                           │
│                                                                                                                 │
│  The local PDF is: Attention Is All You Need (Vaswani et al., 2017), the paper that introduced the              │
│  Transformer.                                                                                                   │
│  Its sections: Attention Is All You Need; Abstract; 1 Introduction; 2 Background; 3 Model Architecture; 3.1     │
│  Encoder and Decoder Stacks; 3.2 Attention; Scaled Dot-Product Attention; 3.2.1 Scaled Dot-Product Attention;   │
│  3.2.2 Multi-Head Attention; 3.2.3 Applications of Attention in our Model; 3.3 Position-wise Feed-Forward       │
│  Networks; 3.4 Embeddings and Softmax; 3.5 Positional Encoding; 4 Why Self-Attention; 5 Training; 5.1 Training  │
│  Data and Batching; 5.2 Hardware and Schedule; 5.3 Optimizer; 5.4 Regularization; 6 Results; 6.1 Machine        │
│  Translation; 6.2 Model Variations; 6.3 English Constituency Parsing; 7 Conclusion; References; Input-Input     │
│  Layer5                                                                                                         │
│                                                                                                                 │
│  Classify this question and respond with EXACTLY ONE of these labels, checking IN ORDER:                        │
│    pdf_search  — the PDF covers this topic (prefer this even if you already know the answer)                    │
│    web_search  — the answer requires current internet information                                               │
│    generate    — none of the above; general knowledge is enough                                                 │
│                                                                                                                 │
│  Do NOT include any explanation, punctuation, or extra words — just the label.                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: The Router Agent has provided a routing decision (see context above).                                    │
│  The original user question is:                                                                                 │
│  "What is the capital of France?"                                                                               │
│                                                                                                                 │
│  Follow the routing decision precisely:                                                                         │
│    - pdf_search -> call the pdf_search tool with the question as input                                          │
│    - web_search -> call the web_search tool with the question as input                                          │
│    - generate   -> answer directly from your LLM knowledge                                                      │
│                                                                                                                 │
│  Compose a comprehensive, well-structured answer.                                                               │
│  Clearly state which source (PDF, Web, or LLM knowledge) was used.                                              │
│                                                                                                                 │
│  Agent: Information Retriever                                                                                   │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

2026-10-07 07:45:45,437 [INFO] Route chosen : generate


╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: b60e63f2-a7f0-42e1-ba22-6edd87835e29                                                                       │
│  Final Output: Based on the routing decision "generate", I will answer this question directly from my           │
│  knowledge.                                                                                                     │
│                                                                                                                 │
│  **Answer:**                                                                                                    │
│                                                                                                                 │
│  The capital of France is **Paris**.                                                                            │
│                                                                                                                 │
│  Paris has been the capital of France since the 12th century and is located in the north-central part of the    │
│  country along the Seine River. It is not only the political and administrative center of France but also its   │
│  largest city and a major global center for art, fashion, gastronomy, and culture. The city is home to          │
│  important government institutions including the Élysée Palace (the official residence of the French            │
│  President) and the French Parliament.                                                                          │
│                                                                                                                 │
│  **Source:** This answer is provided from my built-in knowledge base (LLM knowledge), as indicated by the       │
│  "generate" routing decision.                                                                                   │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

2026-10-07 07:45:45,440 [INFO] Final answer : Based on the routing decision "generate", I will answer this question directly from my knowledge.

**Answer:**

The capital of France is **Paris**.

Paris has been the capital of France since the 12th century and is located in the north-central part of the country along the Seine River. It is not on...
2026-10-07 07:45:45,441 [INFO] ============================================================



📍 Route  : generate
💬 Answer : Based on the routing decision "generate", I will answer this question directly from my knowledge.

**Answer:**

The capital of France is **Paris**.

Paris has been the capital of France since the 12th century and is located in the north-central part of the country along the Seine River. It is not only the political and administrative center of France but also its largest city and a major global center for art, fashion, gastronomy, and culture. The city is home to important government institution...


## 11. Display Structured Results

In [14]:
import pandas as pd

df = pd.DataFrame([
  {
    "#": i + 1,
    "Question": r["question"],
    "Route": r["route"],
    "Answer (truncated)": r["answer"][:120] + "...",
    "Timestamp": r["timestamp"],
  }
  for i, r in enumerate(results)
])
pd.set_option("display.max_colwidth", 130)
print("\n📊 Results Summary")
print(df.to_string(index=False))


📊 Results Summary
 #                                                                           Question      Route                                                                                                              Answer (truncated)                        Timestamp
 1 How does multi-head attention work in the Transformer, and why use multiple heads? pdf_search   ## Answer: How Multi-Head Attention Works in the Transformer\n\n**Source: PDF - "Attention Is All You Need" (Original Tran... 2026-10-07T07:45:24.263652+00:00
 2                       What are the latest breakthroughs in AI announced this week? web_search   Based on the web search results, here are the latest breakthroughs in AI announced this week:\n\n**Source: Web Search Resu... 2026-10-07T07:45:39.616372+00:00
 3                                                     What is the capital of France?   generate Based on the routing decision "generate", I will answer this question directly from my knowledge.\n\n**Answer:

## 12. Save Results & Display Trace Log

In [15]:
# Save structured results to JSON
results_file = "agentic_rag_results.json"
with open(results_file, "w") as f:
  json.dump(results, f, indent=2)
print(f"✅ Results saved to: {results_file}")

# Print the reasoning trace log
print(f"\n{'='*70}")
print("📋 REASONING TRACE LOG")
print('='*70)
with open(LOG_FILE, "r") as f:
  print(f.read())

✅ Results saved to: agentic_rag_results.json

📋 REASONING TRACE LOG
2026-10-07 07:42:03,054 [INFO] Logger initialised — Agentic RAG session started.
2026-10-07 07:42:07,934 [INFO] Claude LLM configured: claude-sonnet-4-5-20250929
2026-10-07 07:42:10,033 [INFO] NumExpr defaulting to 2 threads.
2026-10-07 07:42:21,848 [INFO] [Docling] Parsing PDF: transformer_research_paper-dataset.pdf
2026-10-07 07:42:22,496 [INFO] detected formats: [<InputFormat.PDF: 'pdf'>]
2026-10-07 07:42:22,588 [INFO] Going to convert document batch...
2026-10-07 07:42:22,590 [INFO] Initializing pipeline for StandardPdfPipeline with options hash b023c96447cfca452ff1bd014d549b6e
2026-10-07 07:42:22,678 [INFO] Loading plugin 'docling_defaults'
2026-10-07 07:42:22,681 [INFO] Registered picture descriptions: ['picture_description_vlm_engine', 'vlm', 'api']
2026-10-07 07:42:22,767 [INFO] Loading plugin 'docling_defaults'
2026-10-07 07:42:22,795 [INFO] Registered ocr engines: ['auto', 'easyocr', 'kserve_v2_ocr', 'nemotro

## 13. Interactive Query Interface

Change the question below and re-run to test any query through the pipeline.

In [16]:
custom_question = "What BLEU score did the Transformer achieve on English-to-German translation?"

print(f"\n🔍 Running custom query: {custom_question}\n")
custom_result = await run_agentic_rag(custom_question)
print(f"\n📍 Route  : {custom_result['route']}")
print(f"💬 Answer :\n{custom_result['answer']}")

2026-10-07 07:48:35,077 [INFO] ============================================================
2026-10-07 07:48:35,079 [INFO] New query received: What BLEU score did the Transformer achieve on English-to-German translation?



🔍 Running custom query: What BLEU score did the Transformer achieve on English-to-German translation?



╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 14c68a71-188d-4628-94ac-812777a8a3f3                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: A user has asked the following question:                                                                 │
│  "What BLEU score did the Transformer achieve on English-to-German translation?"                                │
│                                                                                                                 │
│  The local PDF is: Attention Is All You Need (Vaswani et al., 2017), the paper that introduced the              │
│  Transformer.                                                                                                   │
│  Its sections: Attention Is All You Need; Abstract; 1 Introduction; 2 Background; 3 Model Architecture; 3.1     │
│  Encoder and Decoder Stacks; 3.2 Attention; Scaled Dot-Product Attention; 3.2.1 Scaled Dot-Product Attention;   │
│  3.2.2 Multi-Head Attention; 3.2.3 Applications of Attention in our Model; 3.3 Position-wise Feed-Forward       │
│  Networks; 3.4 Embeddings and Softmax; 3.5 Positional Encoding; 4 Why Self-Attention; 5 Training; 5.1 Training  │
│  Data and Batching; 5.2 Hardware and Schedule; 5.3 Optimizer; 5.4 Regularization; 6 Results; 6.1 Machine        │
│  Translation; 6.2 Model Variations; 6.3 English Constituency Parsing; 7 Conclusion; References; Input-Input     │
│  Layer5                                                                                                         │
│                                                                                                                 │
│  Classify this question and respond with EXACTLY ONE of these labels, checking IN ORDER:                        │
│    pdf_search  — the PDF covers this topic (prefer this even if you already know the answer)                    │
│    web_search  — the answer requires current internet information                                               │
│    generate    — none of the above; general knowledge is enough                                                 │
│                                                                                                                 │
│  Do NOT include any explanation, punctuation, or extra words — just the label.                                  │
│                                                                                                                 │
│  ID: 513d1b2c-0477-44b9-85f4-5fe71a9cfa1a                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Query Router                                                                                            │
│                                                                                                                 │
│  Task: A user has asked the following question:                                                                 │
│  "What BLEU score did the Transformer achieve on English-to-German translation?"                                │
│                                                                                                                 │
│  The local PDF is: Attention Is All You Need (Vaswani et al., 2017), the paper that introduced the              │
│  Transformer.                                                                                                   │
│  Its sections: Attention Is All You Need; Abstract; 1 Introduction; 2 Background; 3 Model Architecture; 3.1     │
│  Encoder and Decoder Stacks; 3.2 Attention; Scaled Dot-Product Attention; 3.2.1 Scaled Dot-Product Attention;   │
│  3.2.2 Multi-Head Attention; 3.2.3 Applications of Attention in our Model; 3.3 Position-wise Feed-Forward       │
│  Networks; 3.4 Embeddings and Softmax; 3.5 Positional Encoding; 4 Why Self-Attention; 5 Training; 5.1 Training  │
│  Data and Batching; 5.2 Hardware and Schedule; 5.3 Optimizer; 5.4 Regularization; 6 Results; 6.1 Machine        │
│  Translation; 6.2 Model Variations; 6.3 English Constituency Parsing; 7 Conclusion; References; Input-Input     │
│  Layer5                                                                                                         │
│                                                                                                                 │
│  Classify this question and respond with EXACTLY ONE of these labels, checking IN ORDER:                        │
│    pdf_search  — the PDF covers this topic (prefer this even if you already know the answer)                    │
│    web_search  — the answer requires current internet information                                               │
│    generate    — none of the above; general knowledge is enough                                                 │
│                                                                                                                 │
│  Do NOT include any explanation, punctuation, or extra words — just the label.                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Information Retriever                                                                                   │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ## Answer                                                                                                      │
│                                                                                                                 │
│  **Source: PDF - "Attention Is All You Need" (Transformer paper)**                                              │
│                                                                                                                 │
│  According to the original Transformer paper, the model achieved a **BLEU score of 28.4** on the **WMT 2014     │
│  English-to-German translation task**.                                                                          │
│                                                                                                                 │
│  Specifically, the paper states:                                                                                │
│                                                                                                                 │
│  - The **big Transformer model** achieved **28.4 BLEU** on English-to-German translation, which was a new       │
│  state-of-the-art result at the time                                                                            │
│  - This score outperformed the best previously reported models (including ensembles) by more than 2.0 BLEU      │
│  points                                                                                                         │
│  - Even the **base Transformer model** surpassed all previously published models and ensembles at a fraction    │
│  of the training cost                                                                                           │
│  - The big model configuration took 3.5 days to train on 8 P100 GPUs                                            │
│                                                                                                                 │
│  This result was one of the key achievements highlighted in the paper, demonstrating that the Transformer       │
│  architecture based solely on attention mechanisms could outperform existing recurrent and convolutional        │
│  approaches for machine translation tasks.                                                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: The Router Agent has provided a routing decision (see context above).                                    │
│  The original user question is:                                                                                 │
│  "What BLEU score did the Transformer achieve on English-to-German translation?"                                │
│                                                                                                                 │
│  Follow the routing decision precisely:                                                                         │
│    - pdf_search -> call the pdf_search tool with the question as input                                          │
│    - web_search -> call the web_search tool with the question as input                                          │
│    - generate   -> answer directly from your LLM knowledge                                                      │
│                                                                                                                 │
│  Compose a comprehensive, well-structured answer.                                                               │
│  Clearly state which source (PDF, Web, or LLM knowledge) was used.                                              │
│                                                                                                                 │
│  Agent: Information Retriever                                                                                   │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

2026-10-07 07:48:43,689 [INFO] Route chosen : pdf_search
2026-10-07 07:48:43,690 [INFO] Final answer : ## Answer

**Source: PDF - "Attention Is All You Need" (Transformer paper)**

According to the original Transformer paper, the model achieved a **BLEU score of 28.4** on the **WMT 2014 English-to-German translation task**.

Specifically, the paper states:

- The **big Transformer model** achieved **...
2026-10-07 07:48:43,691 [INFO] ============================================================



📍 Route  : pdf_search
💬 Answer :
## Answer

**Source: PDF - "Attention Is All You Need" (Transformer paper)**

According to the original Transformer paper, the model achieved a **BLEU score of 28.4** on the **WMT 2014 English-to-German translation task**.

Specifically, the paper states:

- The **big Transformer model** achieved **28.4 BLEU** on English-to-German translation, which was a new state-of-the-art result at the time
- This score outperformed the best previously reported models (including ensembles) by more than 2.0 BLEU points
- Even the **base Transformer model** surpassed all previously published models and ensembles at a fraction of the training cost
- The big model configuration took 3.5 days to train on 8 P100 GPUs

This result was one of the key achievements highlighted in the paper, demonstrating that the Transformer architecture based solely on attention mechanisms could outperform existing recurrent and convolutional approaches for machine translation tasks.


╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 14c68a71-188d-4628-94ac-812777a8a3f3                                                                       │
│  Final Output: ## Answer                                                                                        │
│                                                                                                                 │
│  **Source: PDF - "Attention Is All You Need" (Transformer paper)**                                              │
│                                                                                                                 │
│  According to the original Transformer paper, the model achieved a **BLEU score of 28.4** on the **WMT 2014     │
│  English-to-German translation task**.                                                                          │
│                                                                                                                 │
│  Specifically, the paper states:                                                                                │
│                                                                                                                 │
│  - The **big Transformer model** achieved **28.4 BLEU** on English-to-German translation, which was a new       │
│  state-of-the-art result at the time                                                                            │
│  - This score outperformed the best previously reported models (including ensembles) by more than 2.0 BLEU      │
│  points                                                                                                         │
│  - Even the **base Transformer model** surpassed all previously published models and ensembles at a fraction    │
│  of the training cost                                                                                           │
│  - The big model configuration took 3.5 days to train on 8 P100 GPUs                                            │
│                                                                                                                 │
│  This result was one of the key achievements highlighted in the paper, demonstrating that the Transformer       │
│  architecture based solely on attention mechanisms could outperform existing recurrent and convolutional        │
│  approaches for machine translation tasks.                                                                      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## 14. Architecture Summary

| Component | Role | Technology |
|-----------|------|------------|
| Router Agent | Classifies query intent and selects retrieval path | CrewAI Agent + **Claude** |
| Retriever Agent | Executes retrieval and synthesises grounded answer | CrewAI Agent + **Claude** |
| Docling parser | Structure-aware PDF → Markdown | **Docling** (v2 API) |
| Local embeddings | Vectorise chunks with no API key | **HF all-MiniLM-L6-v2** |
| Vector store | Store + similarity search | **ChromaDB** (explicit) |
| Web search tool | Live web search for real-time info | Tavily Search API |
| Orchestration | Sequential task execution with context passing | CrewAI `Process.sequential` |
| Logging | Full reasoning trace to file + stdout | Python `logging` |

### Design Trade-offs

- **Local embeddings over a hosted API.** Local `all-MiniLM-L6-v2` has no key, no cost, and no remote endpoint — the most reliable choice for a live demo, at the price of a one-time model download and slightly lower embedding quality than a large hosted model.
- **Explicit Docling pipeline over `PDFSearchTool`.** More code, but every stage (parse → chunk → embed → store → search) is visible and swappable. For a non-technical audience this turns RAG from a magic box into five labeled steps.
- **Docling specifically.** Structure recognition (tables, reading order, multi-column layout) produces cleaner Markdown than a naive text dump, which chunks better and improves retrieval on a dense research paper.
- **Claude for both agents.** A single shared LLM keeps the demo simple; swap in a larger Claude model for the Retriever if you want stronger synthesis than routing.
- **Sequential process.** The Router's label is passed as context to the Retriever, so the two must run in order.
